# Remote Capability Proxy

> Bridge between Host application and isolated Worker processes

In [ ]:
#| default_exp core.proxy

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import logging
import os
import signal
import socket
import subprocess
import sys
import threading
import time
import uuid
from pathlib import Path
from typing import Any, AsyncGenerator, Dict, Generator, List, Optional, Tuple

import httpx

from cjm_substrate.core.config import get_config
from cjm_substrate.core.errors import (
    CapabilityCancelledError,
    CapabilityFatalError,
    CapabilityInputError,
    CapabilityResourceError,
    CapabilityTransientError,
    ResourceShortfall,
    WorkerOOMError,
)
from cjm_substrate.core.capability import ToolCapability
from cjm_substrate.core.wire import (
    ACCOUNTS_HEADER, ENVELOPE_BODY_KEY, FileBackedDTO, get_call_envelope, wire_decode,
)
from cjm_substrate.core.journal_store import (
    JournalEvent, JournalStore, LocalJournalStore, SubstrateEventType,
)
from cjm_substrate.core.diagnostics_store import (
    DiagnosticsStore, LocalDiagnosticsStore, StreamChunk, normalize_stream_line,
)
from cjm_substrate.core.platform import get_popen_isolation_kwargs, is_windows, terminate_process

# CR-3 follow-up: module-level logger so the proxy can log close-to-wire
# misconfiguration signals (e.g. 404 on /get_system_status meaning the
# substrate wired a non-monitor capability as system_monitor) at ERROR level —
# louder than the substrate's catch-all WARNING degradation path.
_logger = logging.getLogger(__name__)

The `RemoteCapabilityProxy` implements `ToolCapability` but forwards all calls over HTTP to a Worker process running in an isolated environment.

```
Host Application                        Isolated Environment
┌────────────────────┐                 ┌─────────────────────┐
│  CapabilityManager     │                 │  Conda Env (GPU)    │
│    │               │                 │    │                │
│    ▼               │     HTTP        │    ▼                │
│  RemoteCapabilityProxy │◄───────────────▶│  Universal Worker   │
│  (implements       │   localhost     │    │                │
│   ToolCapability) │                 │    ▼                │
│                    │                 │  Actual Capability      │
└────────────────────┘                 └─────────────────────┘
```

Key responsibilities:

1. **Process Management**: Launch worker subprocess with correct Python interpreter
2. **Port Allocation**: Find free port, pass to worker via CLI
3. **Zero-Copy Transfer**: Detect `FileBackedDTO` objects and serialize to temp files
4. **Dual Interface**: Sync methods for scripts, async methods for FastHTML

## RemoteCapabilityProxy

In [ ]:
#| export
from fastcore.basics import patch


In [ ]:
#| export
class RemoteCapabilityProxy(ToolCapability):
    """Proxy that forwards capability calls to an isolated Worker subprocess."""
    
    def __init__(
        self,
        manifest:Dict[str, Any], # Capability manifest with python_path, module, class, etc.
        extra_env:Optional[Dict[str, str]]=None, # CR-12: resolved worker-env overlay (secrets + visible overrides) injected at spawn
        adapter_specs:Optional[List[str]]=None, # CR-17 pt 2: host-matched adapter impl specs ("module:ClassName") bound in-worker at spawn
        journal:Optional[JournalStore]=None, # CR-14: journal sink for worker-lifecycle events; lazy LocalJournalStore at cfg.journal_db_path when None
        diagnostics:Optional[DiagnosticsStore]=None # CR-14: diagnostics sink (raw-stream pump + worker env contract); lazy LocalDiagnosticsStore when None
    ):
        """Initialize proxy and start the worker process."""
        self.manifest = manifest
        self.adapter_specs = list(adapter_specs or [])
        # CR-12: substrate-composed worker-env overlay — resolved secrets (from
        # the SecretStore) + any visible env overrides. Merged into the worker
        # subprocess env at spawn, AFTER the manifest's static env_vars but
        # BEFORE the CJM paths (so an override/secret beats a manifest default,
        # while substrate-owned CJM paths always win). These are fixed at spawn;
        # changing any of them requires a worker RESPAWN (reload_capability).
        self.extra_env = dict(extra_env or {})
        self.process: Optional[subprocess.Popen] = None
        # CR-14: observability sinks. The journal is the host-written
        # account-of-action (worker spawn/ready/death derive from THIS
        # boundary — no capability cooperation involved); diagnostics receives
        # the raw-stream pump's chunks and its path rides the worker env.
        # Lazy defaults share the manager's DB files via get_config().
        cfg = get_config()
        self.journal: JournalStore = journal if journal is not None else LocalJournalStore(cfg.journal_db_path)
        self.diagnostics: DiagnosticsStore = diagnostics if diagnostics is not None else LocalDiagnosticsStore(cfg.diagnostics_db_path)
        self.worker_session_id: Optional[str] = None  # Set per spawn in _start_process
        self._pump_thread: Optional[threading.Thread] = None
        # SG-4: bind the listening socket in the parent and (on Unix) pass the
        # FD to the worker via subprocess inheritance, closing the
        # bind-then-close-then-reopen TOCTOU race that would otherwise let
        # another process steal the port between parent close and worker bind.
        self._listen_sock, self.port = self._bind_listen_socket()
        self.base_url = f"http://127.0.0.1:{self.port}"
        self._start_process()

    @property
    def name(self) -> str: # Capability name from manifest
        """Capability name."""
        return self.manifest.get('name', 'unknown')
    
    @property
    def version(self) -> str: # Capability version from manifest
        """Capability version."""
        return self.manifest.get('version', '0.0.0')

    def _journal_event(
        self,
        event_type:str, # SubstrateEventType value
        payload:Optional[Dict[str, Any]]=None # Per-event structured detail
    ) -> None:
        """Append a worker-lifecycle journal event with this proxy's identity."""
        self.journal.append(JournalEvent(
            event_type=event_type,
            capability_name=self.name,
            worker_session_id=self.worker_session_id,
            payload=payload or {},
        ))

    def initialize(
        self,
        config:Optional[Dict[str, Any]]=None # Configuration dictionary
    ) -> None:
        """Initialize or reconfigure the capability."""
        with httpx.Client() as client:
            resp = client.post(f"{self.base_url}/initialize", json=config or {})
        if resp.status_code != 200:
            raise RuntimeError(f"Initialize failed: {resp.text}")
    
    def execute(
        self,
        *args,
        **kwargs
    ) -> Any: # Capability result
        """Execute the capability synchronously.
        
        CR-4: HTTP 409 from the worker is mapped to a typed
        `CapabilityCancelledError` raised in the host process, so substrate /
        JobQueue / consumer callers can distinguish cooperative cancellation
        from a real capability failure (500 → RuntimeError as before).
        """
        payload = self._prepare_payload(args, kwargs)
        # timeout=None for long-running AI tasks
        with httpx.Client(timeout=None) as client:
            resp = client.post(f"{self.base_url}/execute", json=payload)
        
        if resp.status_code == 409:
            # CR-4: cooperative cancellation surfaced by worker. Re-raise the
            # typed exception so substrate's category-aware retry logic
            # (default_retriable=False) and the JobQueue's cancellation
            # rendering can both see it.
            raise CapabilityCancelledError(self.name)
        if resp.status_code != 200:
            _raise_typed_execute_error(resp, self.name)
        return wire_decode(resp.json())  # stage 2: registered DTOs arrive typed; others stay dicts
    
    def get_config_schema(self) -> Dict[str, Any]: # JSON Schema
        """Get the capability's configuration schema."""
        with httpx.Client() as client:
            return client.get(f"{self.base_url}/config_schema").json()
    
    def get_current_config(self) -> Dict[str, Any]: # Current config values
        """Get the capability's current configuration."""
        with httpx.Client() as client:
            return client.get(f"{self.base_url}/config").json()

In [ ]:
#| export
@patch
def _bind_listen_socket(self:RemoteCapabilityProxy) -> Tuple[socket.socket, int]:
    """Bind a listening socket on a kernel-chosen ephemeral port.
    The socket is kept open so its FD can be inherited by the worker
    subprocess (Unix). Returns (socket, port)."""
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.bind(('127.0.0.1', 0))
    sock.listen(128)
    return sock, sock.getsockname()[1]

In [ ]:
#| export
def _pump_stream(
    stream,  # The worker's merged stdout/stderr pipe (binary)
    diagnostics: DiagnosticsStore,  # Sink for stream chunks
    worker_session_id: str,  # Session attribution for every chunk
) -> None:
    """Pump a worker's raw output to the diagnostics store (CR-14).

    The zero-cooperation death-rattle floor: captures everything the worker
    process writes outside the structured handler — bare prints, native-lib
    output, tqdm, argparse/startup failures BEFORE logging exists, and the
    final traceback of a hard crash. Runs as a daemon thread; ends at EOF
    (worker exit). Attribution is the worker SESSION, never a job — raw
    streams cannot be job-attributed honestly under same-worker concurrency
    (the stage-3 lesson that killed the timestamp-window heuristic).

    tqdm CR-frames collapse to their final frame via `normalize_stream_line`
    (liveness telemetry is not durable). Failures degrade to dropping chunks
    (diagnostics are the disposable class) — never to breaking the worker.
    """
    try:
        for raw in iter(stream.readline, b""):
            try:
                line = normalize_stream_line(raw.decode("utf-8", errors="replace"))
                if line is None:
                    continue
                diagnostics.append_chunk(StreamChunk(
                    content=line, worker_session_id=worker_session_id))
            except Exception:
                continue  # disposable class: drop, never break the pump
    except Exception:
        pass  # pipe closed mid-read (worker kill) — EOF semantics
    finally:
        try:
            stream.close()
        except Exception:
            pass

In [ ]:
#| export
@patch
def _start_process(self:RemoteCapabilityProxy) -> None:
    """Launch the worker subprocess (CR-14: PIPE-captured + journaled).

    Replaces the pre-CR-14 fd-inherited flat log file (`.cjm/logs/<name>.log`
    + ctime session markers): raw output goes through `_pump_stream` into the
    diagnostics store; structured worker logging writes the diagnostics store
    DIRECTLY via the env contract below; the spawn itself is a journal event.
    """
    python_path = self.manifest['python_path']
    cfg = get_config()

    # CR-14: spawn-scoped worker session id — ties WORKER_SPAWNED/READY/DIED
    # journal events and every diagnostics row from this worker lifetime
    # together (the durable replacement for "--- Starting ---" markers).
    self.worker_session_id = str(uuid.uuid4())

    # SG-4: prefer FD inheritance over bind-then-close. `pass_fds` is
    # Unix-only; Windows falls back to the legacy port-handoff (TOCTOU
    # latent there until cross-platform CI matrix per SG-32).
    popen_kwargs: Dict[str, Any] = {}
    if is_windows():
        self._listen_sock.close()
        self._listen_sock = None
        port_args = ["--port", str(self.port)]
    else:
        fd = self._listen_sock.fileno()
        os.set_inheritable(fd, True)
        popen_kwargs["pass_fds"] = (fd,)
        port_args = ["--fd", str(fd)]

    cmd = [
        python_path,
        "-m", "cjm_substrate.core.worker",
        "--module", self.manifest['module'],
        "--class", self.manifest['class'],
        *port_args,
        "--ppid", str(os.getpid())  # Enable suicide pact
    ]
    # CR-17 pt 2: bind host-matched adapter impls in-worker at spawn.
    if getattr(self, 'adapter_specs', None):
        cmd += ["--adapters", ",".join(self.adapter_specs)]

    # Merge environment variables from manifest
    env = dict(os.environ)
    env.update(self.manifest.get('env_vars', {}))
    # CR-12: apply the substrate-composed worker-env overlay (resolved
    # secrets + visible overrides) on top of the manifest defaults.
    env.update(self.extra_env)

    # Inject CJM paths for capability runtime
    env["CJM_CAPABILITY_DATA_DIR"] = str(cfg.capability_data_dir)
    # Stage 8 (PILLAR 1c): the substrate OWNS the per-capability data-dir
    # convention (formerly duplicated in every capability's meta.py). Inject the
    # resolved per-capability dir + ensure it exists, so adapters/tools read
    # CAPABILITY_DATA_DIR instead of recomputing CJM_CAPABILITY_DATA_DIR/<name>.
    _pdd = Path(cfg.capability_data_dir) / self.name
    _pdd.mkdir(parents=True, exist_ok=True)
    env["CAPABILITY_DATA_DIR"] = str(_pdd)
    if cfg.models_dir:
        env["CJM_MODELS_DIR"] = str(cfg.models_dir)
    # CR-14: worker-side diagnostics contract (install_worker_diagnostics):
    # the worker's root logger writes structured records straight to the
    # diagnostics store, stamped with this session id + (via the call
    # envelope) exact job identity. Only injectable for path-backed sinks;
    # a custom sink without db_path leaves the worker on its stdout
    # fallback, which the pump still captures as chunks.
    diag_db = getattr(self.diagnostics, 'db_path', None)
    if diag_db is not None:
        env["CJM_DIAGNOSTICS_DB"] = str(diag_db)
    env["CJM_WORKER_SESSION_ID"] = self.worker_session_id

    print(f"[{self.name}] Starting worker on port {self.port}...", file=sys.stderr)
    print(f"[{self.name}] Diagnostics: {diag_db or self.diagnostics} (session {self.worker_session_id[:8]})", file=sys.stderr)

    # Get cross-platform process isolation kwargs
    isolation_kwargs = get_popen_isolation_kwargs()

    # CR-14: raw stdout/stderr -> PIPE -> pump thread -> diagnostics chunks.
    self.process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, # Merge stderr into stdout
        env=env,
        **isolation_kwargs,
        **popen_kwargs,
    )
    self._pump_thread = threading.Thread(
        target=_pump_stream,
        args=(self.process.stdout, self.diagnostics, self.worker_session_id),
        name=f"cjm-pump-{self.name}",
        daemon=True,
    )
    self._pump_thread.start()

    # CR-14: the spawn is a journal event — derived at the host boundary,
    # zero capability cooperation. Secrets stay out: overlay KEY NAMES only.
    self._journal_event(SubstrateEventType.WORKER_SPAWNED.value, {
        "pid": self.process.pid,
        "port": self.port,
        "module": self.manifest.get('module'),
        "version": self.version,
        "env_overlay_keys": sorted(self.extra_env),
    })
    if self.adapter_specs:
        self._journal_event(SubstrateEventType.ADAPTER_BOUND.value, {
            "specs": list(self.adapter_specs),
        })

    # Parent releases its copy of the listening socket once the worker has
    # inherited the FD; the worker now owns it via uvicorn's --fd path.
    if self._listen_sock is not None:
        self._listen_sock.close()
        self._listen_sock = None
    self._wait_for_ready()

In [ ]:
#| export
@patch
def _wait_for_ready(
    self:RemoteCapabilityProxy,
    timeout:float=30.0 # Max seconds to wait for worker startup
) -> None:
    """Wait for worker to become responsive."""
    start = time.time()
    while time.time() - start < timeout:
        try:
            with httpx.Client() as client:
                client.get(f"{self.base_url}/health")
            print(f"[{self.name}] Worker ready.", file=sys.stderr)
            # CR-14: readiness is a journal event (startup latency rides along).
            self._journal_event(SubstrateEventType.WORKER_READY.value, {
                "wait_seconds": round(time.time() - start, 3),
            })
            return
        except httpx.ConnectError:
            time.sleep(0.5)

    # Timeout. The death rattle (argparse/import/startup failures) was pumped
    # into the diagnostics stream chunks — point the operator at the session.
    # (Pre-CR-14 this called process.communicate(), which now would race the
    # pump thread for the pipe.)
    self._journal_event(SubstrateEventType.WORKER_DIED.value, {
        "phase": "startup_timeout",
        "timeout_seconds": timeout,
        "returncode": self.process.poll() if self.process else None,
    })
    rattle = ""
    try:
        chunks = self.diagnostics.query_chunks(
            worker_session_id=self.worker_session_id, limit=5, descending=True)
        rattle = " | ".join(c.content for c in reversed(chunks))
    except Exception:
        pass
    raise TimeoutError(
        f"Capability '{self.name}' failed to start within {timeout}s "
        f"(worker_session_id={self.worker_session_id}). "
        f"Last output: {rattle or '<no output captured>'}"
    )

In [ ]:
#| export
@patch
def config_options(self:RemoteCapabilityProxy) -> Dict[str, Any]: # CR-11: live config option domains
    """Get the capability's runtime config option providers (CR-11).

    Returns the worker's get_config_options() output (FieldOptions per
    dynamic field, JSON-serialized to dicts). Empty dict when the capability
    exposes no dynamic options.
    """
    with httpx.Client() as client:
        return client.get(f"{self.base_url}/config_options").json()

In [ ]:
#| export
@patch
def cleanup(self:RemoteCapabilityProxy) -> None:
    """Clean up capability resources and terminate worker process."""
    # Send cleanup request to worker
    try:
        with httpx.Client(timeout=2) as client:
            client.post(f"{self.base_url}/cleanup")
    except Exception:
        pass  # Worker may already be gone

    # Terminate the subprocess using cross-platform utility
    proc = self.process
    terminate_process(proc, timeout=2.0)
    self.process = None

    # CR-14: journal the death. Documented exception to the journal's loud
    # rule: cleanup runs in teardown paths (context-manager __exit__,
    # interpreter shutdown) where a raise would mask the work that ran —
    # so a failed death-record logs at ERROR (visible, not silent) instead
    # of raising. Every other journal append stays loud.
    try:
        self._journal_event(SubstrateEventType.WORKER_DIED.value, {
            "phase": "cleanup",
            "returncode": proc.returncode if proc is not None else None,
        })
    except Exception:
        _logger.error(
            "journal append failed recording WORKER_DIED for %s (session %s)",
            self.name, self.worker_session_id, exc_info=True)

    # The pump thread ends at pipe EOF (worker exit); give it a moment to
    # flush the final chunks. Daemon thread — never blocks shutdown long.
    if self._pump_thread is not None:
        self._pump_thread.join(timeout=2.0)
        self._pump_thread = None

### Input Serialization

The proxy detects `FileBackedDTO` objects and serializes them to temp files before transmission. This enables zero-copy transfer of large data (audio, images) between processes.

In [ ]:
#| export
def _maybe_serialize_input(
    self,
    obj: Any # Object to potentially serialize
) -> Any: # Serialized form (path string or original object)
    """Convert FileBackedDTO objects to file paths for zero-copy transfer."""
    # Zero-Copy: Ask object to save itself
    if isinstance(obj, FileBackedDTO):
        return obj.to_temp_file()
    # Paths: Convert to string
    if isinstance(obj, Path):
        return str(obj)
    # Default: Pass through for JSON serialization
    return obj

def _prepare_payload(
    self,
    args: tuple, # Positional arguments
    kwargs: dict # Keyword arguments
) -> Dict[str, Any]: # JSON-serializable payload
    """Prepare arguments for HTTP transmission.

    CR-14: attaches the current call envelope (set by the JobQueue around
    each job's execution via the `wire` contextvar) as a TOP-LEVEL body key.
    Never inside kwargs — capability signatures never see it; old workers ignore
    unknown top-level keys. Envelope-less calls (direct proxy use) simply
    produce unattributed worker records.
    """
    safe_args = [self._maybe_serialize_input(a) for a in args]
    safe_kwargs = {k: self._maybe_serialize_input(v) for k, v in kwargs.items()}
    payload: Dict[str, Any] = {"args": safe_args, "kwargs": safe_kwargs}
    env = get_call_envelope()
    if env is not None:
        payload[ENVELOPE_BODY_KEY] = env.to_wire()
    return payload

RemoteCapabilityProxy._maybe_serialize_input = _maybe_serialize_input
RemoteCapabilityProxy._prepare_payload = _prepare_payload

In [ ]:
#| export
def _harvest_worker_accounts(
    self,
    resp,  # The worker's httpx response (any status — error paths report too)
) -> None:
    """CR-14 follow-up: journal in-worker accounts off the response header.

    The host-writes-the-row half of the account contract (`wire.
    record_account` / worker `_accounts_headers`): workers RECORD accounts
    during a call span; the proxy journals them ON RECEIPT with
    `worker_reported=True` + the receiving-side identity (the proxy-side
    call envelope + this spawn's worker session). Called on every unary
    response BEFORE the status checks so a `_job_error` 500's accounts
    (e.g. a save that succeeded before a later crash) are kept.

    Header absent (old workers, account-less calls) = no-op. A failure here
    logs at ERROR and never breaks the call — the result is the contract;
    and a wedged journal store also fails the queue's own emission for the
    same job, so the wedge gate still fires loudly.
    """
    raw = resp.headers.get(ACCOUNTS_HEADER)
    if not raw:
        return
    try:
        env = get_call_envelope()
        for acct in json.loads(raw):
            self.journal.append(JournalEvent(
                event_type=str(acct.get("event_type") or "worker_account"),
                run_id=env.run_id if env is not None else None,
                job_id=env.job_id if env is not None else None,
                composition_id=env.composition_id if env is not None else None,
                node_id=env.node_id if env is not None else None,
                actor=env.actor if env is not None else None,
                capability_name=self.name,
                worker_session_id=self.worker_session_id,
                worker_reported=True,
                payload=acct.get("payload") or {},
            ))
    except Exception:
        _logger.error(
            "failed to journal worker accounts for %s (session %s)",
            self.name, self.worker_session_id, exc_info=True)

RemoteCapabilityProxy._harvest_worker_accounts = _harvest_worker_accounts

In [ ]:
#| hide
# CR-14 follow-up: _harvest_worker_accounts journals header-borne accounts
# with worker_reported=True + the proxy-side call-envelope identity; header
# absent = no-op; malformed header = ERROR-logged, never a raise.
from cjm_substrate.core.wire import (
    ACCOUNTS_HEADER as _AH_h, CallEnvelope as _CE_h,
    reset_call_envelope as _rce_h, set_call_envelope as _sce_h,
)


class _ListJournal:
    def __init__(self):
        self.rows = []

    def append(self, ev):
        self.rows.append(ev)
        return len(self.rows)


class _HarvestStubProxy:
    name = "stub-capability"
    worker_session_id = "ws-h1"
    def __init__(self):
        self.journal = _ListJournal()
    _harvest_worker_accounts = _harvest_worker_accounts


class _HdrResp:
    def __init__(self, headers):
        self.headers = headers


p = _HarvestStubProxy()

# (a) Header absent -> no rows.
p._harvest_worker_accounts(_HdrResp({}))
assert p.journal.rows == []

# (b) Accounts journaled with envelope identity + worker_reported=True.
tok = _sce_h(_CE_h(job_id="j-h", run_id="r-h", actor="cli:t"))
try:
    p._harvest_worker_accounts(_HdrResp({_AH_h: json.dumps([
        {"event_type": "result_saved", "payload": {"row_job_id": "row-9"}},
        {"event_type": "task_account", "payload": {"task": "t", "ok": True}},
    ])}))
finally:
    _rce_h(tok)
assert len(p.journal.rows) == 2
r0 = p.journal.rows[0]
assert r0.event_type == "result_saved" and r0.worker_reported is True
assert r0.job_id == "j-h" and r0.run_id == "r-h" and r0.actor == "cli:t"
assert r0.capability_name == "stub-capability" and r0.worker_session_id == "ws-h1"
assert r0.payload == {"row_job_id": "row-9"}

# (c) Envelope-less harvest: honestly unattributed rows, still journaled.
p2 = _HarvestStubProxy()
p2._harvest_worker_accounts(_HdrResp({_AH_h: json.dumps([
    {"event_type": "cache_hit", "payload": {}}])}))
assert p2.journal.rows[0].job_id is None and p2.journal.rows[0].worker_reported is True

# (d) Malformed header: no raise (ERROR-logged), call survives.
p3 = _HarvestStubProxy()
p3._harvest_worker_accounts(_HdrResp({_AH_h: "{not json"}))
assert p3.journal.rows == []

print("proxy account harvest: PASS")

### Asynchronous Interface

These methods are `async` for use with FastHTML and other async frameworks. Use `execute_stream` for real-time streaming results.

In [ ]:
#| export
async def execute_async(
    self,
    *args,
    **kwargs
) -> Any: # Capability result
    """Execute the capability asynchronously.
    
    CR-4: HTTP 409 from the worker is mapped to a typed `CapabilityCancelledError`.
    Same 409/200/other semantics as the sync `execute()` variant.
    """
    payload = self._prepare_payload(args, kwargs)
    async with httpx.AsyncClient(timeout=None) as client:
        resp = await client.post(f"{self.base_url}/execute", json=payload)
    
    if resp.status_code == 409:
        # CR-4: see sync variant
        raise CapabilityCancelledError(self.name)
    if resp.status_code != 200:
        _raise_typed_execute_error(resp, self.name)
    return wire_decode(resp.json())  # stage 2: registered DTOs arrive typed; others stay dicts

RemoteCapabilityProxy.execute_async = execute_async

In [ ]:
#| export
def _raise_from_job_error_chunk(
    job_error: Dict[str, Any],  # _job_error payload from /execute_stream terminal chunk
    capability_name: str,  # Caller's capability name (for CapabilityCancelledError reconstruction)
) -> None:
    """SG-52: convert a `_job_error` JobError-shaped dict into the right typed exception.
    
    Mapping rules:
    - `original_exc_repr` starts with `"CapabilityCancelledError"` → raise CapabilityCancelledError
      (preserves the non-retriable semantic that category alone doesn't capture).
    - category == "user_input" → CapabilityInputError (with fields_invalid).
    - category == "transient" → CapabilityTransientError (with retry_after_seconds).
    - category == "resource" → CapabilityResourceError (with reconstructed ResourceShortfall).
    - category == "fatal" → CapabilityFatalError.
    - Unknown category → RuntimeError carrying the chunk for forensic inspection.
    
    This is the streaming-side counterpart to /execute's 409 → CapabilityCancelledError
    detection. Same intent: the typed exception survives the HTTP wire boundary
    so substrate / JobQueue / consumer code can branch on category without
    parsing string messages.
    """
    category = job_error.get("category", "fatal")
    message = job_error.get("message", "Capability error in stream")
    repr_str = job_error.get("original_exc_repr", "") or ""
    
    # Cancellation special-case (transient category, non-retriable semantic)
    if repr_str.startswith("CapabilityCancelledError"):
        raise CapabilityCancelledError(capability_name)
    
    if category == "user_input":
        raise CapabilityInputError(message, fields_invalid=job_error.get("fields_invalid"))
    if category == "transient":
        raise CapabilityTransientError(message, retry_after_seconds=job_error.get("retry_after_seconds"))
    if category == "resource":
        shortfall_dict = job_error.get("resource_shortfall")
        shortfall = ResourceShortfall(**shortfall_dict) if shortfall_dict else None
        raise CapabilityResourceError(message, resource_shortfall=shortfall)
    if category == "fatal":
        raise CapabilityFatalError(message)
    # Unknown category — surface as RuntimeError with the structured payload
    raise RuntimeError(f"Capability stream error (unknown category {category!r}): {job_error}")

In [ ]:
#| export
def _raise_typed_execute_error(
    resp,              # The worker's non-200 httpx response
    capability_name: str,  # Capability name for exception context
) -> None:             # Never returns — always raises
    """SG-52 parity for the unary execute path (stage-3 ledger G7).

    If the worker's error body carries the `{"_job_error": <JobError dict>}`
    sentinel (post-fix workers), raise the corresponding TYPED exception via
    `_raise_from_job_error_chunk` — this is what lets the manager's CR-7
    reactive-retry path see `CapabilityResourceError` from plain `/execute`
    calls (the OOM-backstop stress test caught the unary channel collapsing
    every failure to RuntimeError, leaving the retry blind). Pre-fix workers
    return a bare-string detail → the legacy RuntimeError fallback (version-
    skew tolerance, F12 posture).
    """
    try:
        body = resp.json()
    except Exception:
        body = None
    if isinstance(body, dict) and "_job_error" in body:
        _raise_from_job_error_chunk(body["_job_error"], capability_name)
    raise RuntimeError(f"Execute failed: {resp.text}")

In [ ]:
#| hide
# G7 wire-format executable test: the unary execute error contract.
# Post-fix workers return {"_job_error": <JobError dict>} as the 500 body ->
# the proxy must raise the TYPED exception (what CR-7 retry catches);
# pre-fix workers return a bare-string detail -> legacy RuntimeError fallback.
from cjm_substrate.core.errors import (
    CapabilityResourceError as _PRE_t, CapabilityFatalError as _PFE_t,
)


class _FakeResp:
    def __init__(self, body, text="boom"):
        self._body = body
        self.text = text

    def json(self):
        if isinstance(self._body, Exception):
            raise self._body
        return self._body


# (a) resource-category _job_error -> typed CapabilityResourceError (CR-7 visible).
try:
    _raise_typed_execute_error(_FakeResp({"_job_error": {
        "category": "resource", "message": "CUDA OOM loading model",
        "resource_shortfall": {"resource": "gpu_memory_mb", "needed": 48000.0,
                               "available": 23500.0},
    }}), "fake-capability")
    raise AssertionError("expected CapabilityResourceError")
except _PRE_t as e:
    assert "CUDA OOM" in str(e)

# (b) fatal-category _job_error -> CapabilityFatalError.
try:
    _raise_typed_execute_error(_FakeResp({"_job_error": {
        "category": "fatal", "message": "model file corrupt"}}), "fake-capability")
    raise AssertionError("expected CapabilityFatalError")
except _PFE_t:
    pass

# (c) pre-fix worker: bare-string detail -> legacy RuntimeError fallback.
try:
    _raise_typed_execute_error(_FakeResp({"detail": "boom"}, text="boom"), "fake-capability")
    raise AssertionError("expected RuntimeError")
except RuntimeError as e:
    assert "Execute failed" in str(e)

# (d) non-JSON body -> RuntimeError fallback.
try:
    _raise_typed_execute_error(_FakeResp(ValueError("not json"), text="<html>"), "fake-capability")
    raise AssertionError("expected RuntimeError")
except RuntimeError as e:
    assert "Execute failed" in str(e)
print("G7 unary typed-error contract: PASS")

In [ ]:
#| export
def execute_stream_sync(self, *args, **kwargs) -> Generator[Any, None, None]:
    """Synchronous wrapper for streaming (blocking)."""
    # This is tricky without "httpx.stream" in sync mode.
    # For now, it's okay to leave it async-only if documented.
    pass

RemoteCapabilityProxy.execute_stream_sync = execute_stream_sync

In [ ]:
#| export
async def execute_stream(
    self,
    *args,
    **kwargs
) -> AsyncGenerator[Any, None]: # Yields parsed JSON chunks
    """Execute with streaming response (async generator).
    
    SG-52: detects the terminal `{"_job_error": <JobError dict>}` chunk and
    raises the corresponding typed exception client-side instead of yielding
    it to downstream consumers. Mirrors /execute's HTTP 409 → typed-exception
    behavior at the streaming wire boundary. Normal capability output chunks
    pass through unchanged (they never carry the `_job_error` key).
    """
    payload = self._prepare_payload(args, kwargs)
    async with httpx.AsyncClient(timeout=None) as client:
        async with client.stream("POST", f"{self.base_url}/execute_stream", json=payload) as resp:
            async for line in resp.aiter_lines():
                if not line:
                    continue
                chunk = json.loads(line)
                # SG-52: typed error sentinel → raise rather than yield
                if isinstance(chunk, dict) and "_job_error" in chunk:
                    _raise_from_job_error_chunk(chunk["_job_error"], self.name)
                yield wire_decode(chunk)  # stage 2: typed chunks for registered DTOs

RemoteCapabilityProxy.execute_stream = execute_stream

### CR-7 Track A: Worker-death classification

When the worker subprocess dies mid-call, the httpx client fault propagates up. The substrate needs to classify the death to dispatch CR-7's reactive retry path correctly:

| Worker state after httpx fault | Classification | Raised exception |
|---|---|---|
| Still alive | Network blip / timeout — not CR-7's concern | Re-raise original httpx error |
| Dead with `returncode == -SIGKILL` (-9 on POSIX) | Kernel OOM-killer is the dominant cause | `WorkerOOMError` |
| Dead with any other non-zero returncode | Generic crash (segfault, capability __init__ panic, etc.) | `CapabilityTransientError` |

This is Track A — the substrate-side view. Track B (capability-side, per SG-47's sub-task) catches `torch.cuda.OutOfMemoryError` inside `execute()` and re-raises as `CapabilityResourceError` with a populated `ResourceShortfall` before the worker ever dies; that path doesn't hit Track A at all. Both tracks converge at `except CapabilityResourceError` in CR-7's reactive retry loop.

The classification only applies to the `/execute` and `/execute_stream` paths — the long-running ones. Hook calls (`/on_disable`, `/prefetch`, etc.) already short-circuit on `httpx.ConnectError` and return False without classification, because their callers (CR-2 / CR-4 lifecycle bookkeeping) don't need typed retry.

In [ ]:
#| export
def _check_worker_death(self) -> None:
    """CR-7 Track A: inspect subprocess state after an httpx fault on /execute.
    
    Raises `WorkerOOMError` if the worker died with SIGKILL (POSIX returncode -9 —
    kernel OOM-killer is the dominant cause). Raises `CapabilityTransientError` if
    it died with any other returncode. Returns None silently when the worker
    is still alive (caller re-raises the original httpx error).
    
    `signal.SIGKILL` doesn't exist on Windows, where worker-OOM looks different
    (STATUS_NO_MEMORY 0xC0000017 = -1073741799). Cross-platform OOM detection is
    a future enhancement when there's a concrete Windows substrate consumer.
    """
    if self.process is None:
        # Already cleaned up; nothing to classify
        return
    rc = self.process.poll()
    if rc is None:
        return  # Still alive — original httpx error wasn't worker death
    # Worker is dead. Classify the returncode.
    sigkill_rc = -getattr(signal, "SIGKILL", 9)
    if rc == sigkill_rc:
        raise WorkerOOMError(self.name, process_returncode=rc)
    raise CapabilityTransientError(
        f"Worker for capability {self.name!r} died unexpectedly (returncode={rc})"
    )


# Wrapped execute methods — Track A classification on httpx faults.
# The bodies are intentionally near-verbatim copies of the originals + a
# try/except shell; patching overrides the prior definitions on
# RemoteCapabilityProxy. Httpx exceptions caught: ConnectError (worker stopped
# accepting connections), RemoteProtocolError (worker died mid-response),
# ReadError / WriteError (socket reset mid-payload). Other httpx exceptions
# (timeout etc.) pass through unchanged.
def execute_with_oom_check(self, *args, **kwargs) -> Any:
    """CR-7 Track A wrapper around the sync execute path.
    
    Catches httpx connection / protocol faults, calls `_check_worker_death`,
    and either raises a typed `WorkerOOMError` / `CapabilityTransientError` or
    re-raises the original httpx error (worker still alive — caller treats
    as a generic transient network issue).
    """
    payload = self._prepare_payload(args, kwargs)
    try:
        with httpx.Client(timeout=None) as client:
            resp = client.post(f"{self.base_url}/execute", json=payload)
    except (httpx.ConnectError, httpx.RemoteProtocolError,
            httpx.ReadError, httpx.WriteError):
        # CR-7 Track A: worker may have died mid-call. Classify or re-raise.
        self._check_worker_death()
        raise  # Worker still alive — propagate the original transport fault
    
    # CR-14 follow-up: journal worker-reported accounts BEFORE status checks
    # (a _job_error 500's pre-failure accounts are kept).
    self._harvest_worker_accounts(resp)
    if resp.status_code == 409:
        # CR-4: cooperative cancellation surfaced by worker
        raise CapabilityCancelledError(self.name)
    if resp.status_code != 200:
        _raise_typed_execute_error(resp, self.name)  # G7: typed unary error channel
    return wire_decode(resp.json())  # stage 2: registered DTOs arrive typed; others stay dicts


async def execute_async_with_oom_check(self, *args, **kwargs) -> Any:
    """CR-7 Track A wrapper around the async execute path. Same semantics."""
    payload = self._prepare_payload(args, kwargs)
    try:
        async with httpx.AsyncClient(timeout=None) as client:
            resp = await client.post(f"{self.base_url}/execute", json=payload)
    except (httpx.ConnectError, httpx.RemoteProtocolError,
            httpx.ReadError, httpx.WriteError):
        self._check_worker_death()
        raise
    
    self._harvest_worker_accounts(resp)  # CR-14 follow-up (see sync variant)
    if resp.status_code == 409:
        raise CapabilityCancelledError(self.name)
    if resp.status_code != 200:
        _raise_typed_execute_error(resp, self.name)  # G7: typed unary error channel
    return wire_decode(resp.json())  # stage 2: registered DTOs arrive typed; others stay dicts


RemoteCapabilityProxy._check_worker_death = _check_worker_death
RemoteCapabilityProxy.execute = execute_with_oom_check
RemoteCapabilityProxy.execute_async = execute_async_with_oom_check

In [ ]:
#| export
def _prepare_task_payload(
    self,
    task_name: str,  # Adapter task address
    method: str,  # Adapter method name
    kwargs: dict,  # Task method kwargs
) -> Dict[str, Any]:  # JSON-serializable /task body
    """Build the /task body (CR-17 pt 2) + the CR-14 call envelope rider.

    Same envelope semantics as `_prepare_payload`: top-level key, never
    inside kwargs.
    """
    payload: Dict[str, Any] = {
        "task": task_name,
        "method": method,
        "kwargs": {k: self._maybe_serialize_input(v) for k, v in kwargs.items()},
    }
    env = get_call_envelope()
    if env is not None:
        payload[ENVELOPE_BODY_KEY] = env.to_wire()
    return payload


def execute_task(self, task_name: str, method: str, **kwargs) -> Any:
    """Invoke a typed task-adapter method in-worker (CR-17 pt 2; sync).

    The explicit task channel: `action=` stays the tool's native in-worker
    dispatch; this addresses the TASK contract (adapter + method). Kwargs-only
    by design. Built ONCE with the CR-7 Track-A worker-death check inside —
    no later wrapper supersedes it (the G7a last-assignment-wins lesson).
    """
    payload = self._prepare_task_payload(task_name, method, kwargs)
    try:
        with httpx.Client(timeout=None) as client:
            resp = client.post(f"{self.base_url}/task", json=payload)
    except (httpx.ConnectError, httpx.RemoteProtocolError,
            httpx.ReadError, httpx.WriteError):
        self._check_worker_death()
        raise
    self._harvest_worker_accounts(resp)  # CR-14 follow-up: TASK_ACCOUNT et al
    if resp.status_code == 409:
        raise CapabilityCancelledError(self.name)
    if resp.status_code != 200:
        _raise_typed_execute_error(resp, self.name)  # G7 typed channel from day one
    return wire_decode(resp.json())


async def execute_task_async(self, task_name: str, method: str, **kwargs) -> Any:
    """Invoke a typed task-adapter method in-worker (CR-17 pt 2; async). Same semantics."""
    payload = self._prepare_task_payload(task_name, method, kwargs)
    try:
        async with httpx.AsyncClient(timeout=None) as client:
            resp = await client.post(f"{self.base_url}/task", json=payload)
    except (httpx.ConnectError, httpx.RemoteProtocolError,
            httpx.ReadError, httpx.WriteError):
        self._check_worker_death()
        raise
    self._harvest_worker_accounts(resp)  # CR-14 follow-up: TASK_ACCOUNT et al
    if resp.status_code == 409:
        raise CapabilityCancelledError(self.name)
    if resp.status_code != 200:
        _raise_typed_execute_error(resp, self.name)
    return wire_decode(resp.json())


RemoteCapabilityProxy._prepare_task_payload = _prepare_task_payload
RemoteCapabilityProxy.execute_task = execute_task
RemoteCapabilityProxy.execute_task_async = execute_task_async

In [ ]:
#| hide
# CR-7 Track A: _check_worker_death classification. Tested with stub processes
# (no real worker subprocess needed). Each scenario constructs a minimal
# proxy-shaped object that satisfies the method's attribute reads.

class _StubProcess:
    def __init__(self, returncode):
        self._rc = returncode
    def poll(self):
        return self._rc


class _StubProxy:
    name = "stub-capability"
    def __init__(self, process):
        self.process = process
    _check_worker_death = _check_worker_death


# Worker alive (poll == None) → returns None silently
alive_proxy = _StubProxy(_StubProcess(returncode=None))
alive_proxy._check_worker_death()  # no exception

# Worker is None (already cleaned up) → returns None silently
none_proxy = _StubProxy(None)
none_proxy._check_worker_death()  # no exception

# Worker dead with SIGKILL (-9 on POSIX) → WorkerOOMError
sigkill_rc = -getattr(signal, "SIGKILL", 9)
oom_proxy = _StubProxy(_StubProcess(returncode=sigkill_rc))
caught_oom = False
try:
    oom_proxy._check_worker_death()
except WorkerOOMError as e:
    caught_oom = True
    assert e.capability_name == "stub-capability"
    assert e.process_returncode == sigkill_rc
    assert isinstance(e, CapabilityResourceError), \
        "WorkerOOMError must catch under CapabilityResourceError (CR-7 reactive retry site)"
assert caught_oom, "SIGKILL must classify as WorkerOOMError"

# Worker dead with other signal (e.g. -11 = SIGSEGV) → CapabilityTransientError
crash_proxy = _StubProxy(_StubProcess(returncode=-11))
caught_transient = False
try:
    crash_proxy._check_worker_death()
except CapabilityTransientError as e:
    caught_transient = True
    assert not isinstance(e, WorkerOOMError), \
        "non-SIGKILL deaths must NOT classify as WorkerOOMError"
    assert "returncode=-11" in str(e)
assert caught_transient, "non-SIGKILL death must classify as CapabilityTransientError"

# Worker dead with normal exit code (1) → CapabilityTransientError (capability crashed
# during __init__ or some other non-OOM cause)
exit1_proxy = _StubProxy(_StubProcess(returncode=1))
caught_transient = False
try:
    exit1_proxy._check_worker_death()
except CapabilityTransientError as e:
    caught_transient = True
assert caught_transient, "normal exit code must classify as CapabilityTransientError"

# Verify the wrapped execute paths are the ones patched onto RemoteCapabilityProxy
assert RemoteCapabilityProxy.execute is execute_with_oom_check, \
    "execute must be the CR-7 Track A wrapper"
assert RemoteCapabilityProxy.execute_async is execute_async_with_oom_check, \
    "execute_async must be the CR-7 Track A wrapper"
assert hasattr(RemoteCapabilityProxy, "_check_worker_death"), \
    "_check_worker_death must be patched onto RemoteCapabilityProxy"

print("✓ CR-7 Track A: _check_worker_death + wrapped execute methods")

In [ ]:
#| hide
# SG-52 test: _raise_from_job_error_chunk maps JobError-shaped dicts to typed
# exceptions correctly across all categories. Pure function, no HTTP needed.

def _sg52_chunk_mapper_check():
    # CapabilityCancelledError special-case via original_exc_repr prefix
    try:
        _raise_from_job_error_chunk(
            {"category": "transient", "message": "cancelled",
             "original_exc_repr": "CapabilityCancelledError('whisper cancelled')"},
            capability_name="whisper",
        )
        assert False, "expected CapabilityCancelledError"
    except CapabilityCancelledError as e:
        assert e.capability_name == "whisper"
        assert not isinstance(e, ValueError), "CapabilityCancelledError must NOT be ValueError-catchable"
    
    # user_input → CapabilityInputError with fields_invalid
    try:
        _raise_from_job_error_chunk(
            {"category": "user_input", "message": "bad config",
             "original_exc_repr": "CapabilityConfigError(...)", "fields_invalid": ["model"]},
            capability_name="whisper",
        )
        assert False, "expected CapabilityInputError"
    except CapabilityInputError as e:
        assert not isinstance(e, ValueError), "SG-48 dropped the ValueError base"
        assert e.fields_invalid == ["model"]
        assert "bad config" in str(e)
    
    # transient (non-cancellation) → CapabilityTransientError with retry_after_seconds
    try:
        _raise_from_job_error_chunk(
            {"category": "transient", "message": "network blip",
             "original_exc_repr": "TimeoutError(...)", "retry_after_seconds": 5.0},
            capability_name="whisper",
        )
        assert False, "expected CapabilityTransientError"
    except CapabilityTransientError as e:
        assert not isinstance(e, CapabilityCancelledError), \
            "non-cancellation transient must not be a CapabilityCancelledError"
        assert e.retry_after_seconds == 5.0
    
    # resource → CapabilityResourceError with reconstructed ResourceShortfall
    try:
        _raise_from_job_error_chunk(
            {"category": "resource", "message": "oom",
             "original_exc_repr": "CapabilityResourceError(...)",
             "resource_shortfall": {"resource": "gpu_vram_mb", "needed": 8000.0, "available": 4000.0}},
            capability_name="whisper",
        )
        assert False, "expected CapabilityResourceError"
    except CapabilityResourceError as e:
        assert e.resource_shortfall is not None
        assert e.resource_shortfall.needed == 8000.0
        assert e.resource_shortfall.resource == "gpu_vram_mb"
    
    # resource with no shortfall payload — still raises (shortfall=None)
    try:
        _raise_from_job_error_chunk(
            {"category": "resource", "message": "vague resource issue",
             "original_exc_repr": "CapabilityResourceError(...)"},
            capability_name="whisper",
        )
        assert False, "expected CapabilityResourceError"
    except CapabilityResourceError as e:
        assert e.resource_shortfall is None
    
    # fatal → CapabilityFatalError
    try:
        _raise_from_job_error_chunk(
            {"category": "fatal", "message": "bug",
             "original_exc_repr": "RuntimeError(...)"},
            capability_name="whisper",
        )
        assert False, "expected CapabilityFatalError"
    except CapabilityFatalError as e:
        assert e.category == "fatal"
        assert e.default_retriable is False
    
    # Unknown category → RuntimeError (forensic surface for unexpected wire format)
    try:
        _raise_from_job_error_chunk(
            {"category": "weird_category", "message": "x", "original_exc_repr": "..."},
            capability_name="whisper",
        )
        assert False, "expected RuntimeError"
    except RuntimeError as e:
        assert "weird_category" in str(e)
    except CapabilityError:
        assert False, "unknown category must NOT raise a CapabilityError subclass"


from cjm_substrate.core.errors import CapabilityError  # for the negative assertion above
_sg52_chunk_mapper_check()
print("SG-52 _raise_from_job_error_chunk category dispatch: PASS")

### Lifecycle Management

In [ ]:
    #| export
def get_stats(self) -> Dict[str, Any]: # Process telemetry
    """Get worker process resource usage."""
    with httpx.Client() as client:
        return client.get(f"{self.base_url}/stats").json()

def is_alive(self) -> bool: # True if worker is responsive
    """Check if the worker process is still running and responsive."""
    if not self.process or self.process.poll() is not None:
        return False
    try:
        with httpx.Client(timeout=2) as client:
            resp = client.get(f"{self.base_url}/health")
            return resp.status_code == 200
    except Exception:
        return False

RemoteCapabilityProxy.get_stats = get_stats
RemoteCapabilityProxy.is_alive = is_alive

In [ ]:
#| export
def get_structural_surface(self) -> Optional[Dict[str, Any]]:  # Live-derived surface, or None
    """Pass-2 Thread 3 live companion: fetch the worker's runtime-derived
    structural surface (`GET /structural_surface`).

    Returns None when the worker predates the endpoint (a pre-fracture
    substrate in a snapshot env → 404) or on transport failure — callers
    treat None as "skip the drift check", never as an empty surface.
    """
    try:
        with httpx.Client(timeout=5) as client:
            resp = client.get(f"{self.base_url}/structural_surface")
        if resp.status_code != 200:
            return None
        return resp.json()
    except Exception:
        return None

RemoteCapabilityProxy.get_structural_surface = get_structural_surface

### Cancellation and Progress

Methods for cancelling running executions and polling progress status.

In [ ]:
#| export
def cancel(self) -> bool: # True if cancel request was sent
    """Request cancellation of running execution."""
    try:
        with httpx.Client(timeout=2) as client:
            resp = client.post(f"{self.base_url}/cancel")
        return resp.status_code == 200
    except httpx.ConnectError:
        return False  # Worker may have died

async def cancel_async(self) -> bool: # True if cancel request was sent
    """Request cancellation asynchronously."""
    try:
        async with httpx.AsyncClient(timeout=2) as client:
            resp = await client.post(f"{self.base_url}/cancel")
        return resp.status_code == 200
    except httpx.ConnectError:
        return False

def get_progress(self) -> Dict[str, Any]: # {progress: float, message: str}
    """Get current execution progress from worker."""
    try:
        with httpx.Client(timeout=2) as client:
            resp = client.get(f"{self.base_url}/progress")
        if resp.status_code == 200:
            return resp.json()
        return {"progress": 0.0, "message": ""}
    except httpx.ConnectError:
        return {"progress": 0.0, "message": ""}

async def get_progress_async(self) -> Dict[str, Any]: # {progress: float, message: str}
    """Get current execution progress asynchronously."""
    try:
        async with httpx.AsyncClient(timeout=2) as client:
            resp = await client.get(f"{self.base_url}/progress")
        if resp.status_code == 200:
            return resp.json()
        return {"progress": 0.0, "message": ""}
    except httpx.ConnectError:
        return {"progress": 0.0, "message": ""}

RemoteCapabilityProxy.cancel = cancel
RemoteCapabilityProxy.cancel_async = cancel_async
RemoteCapabilityProxy.get_progress = get_progress
RemoteCapabilityProxy.get_progress_async = get_progress_async

In [ ]:
#| export
def on_disable(self) -> bool:  # True if hook signal accepted by worker
    """CR-2: forward the substrate's on_disable signal to the worker process.
    
    Capability can opt in via ToolCapability.on_disable(); default implementation
    is a no-op so silent-pass-through is the norm. Failures to reach the
    worker (already terminated, network blip) are logged-and-swallowed —
    the substrate-side enable/disable bookkeeping doesn't depend on the
    hook actually firing.
    """
    try:
        with httpx.Client(timeout=5) as client:
            resp = client.post(f"{self.base_url}/on_disable")
        return resp.status_code == 200
    except httpx.ConnectError:
        return False  # Worker may have died; substrate carries on


def on_enable(self) -> bool:  # True if hook signal accepted by worker
    """CR-2: forward the substrate's on_enable signal to the worker process.
    
    Same delivery semantics as on_disable: best-effort, errors logged-and-
    swallowed. The capability's hook (default no-op) decides whether to eagerly
    re-acquire resources or rely on lazy re-load at next execute().
    """
    try:
        with httpx.Client(timeout=5) as client:
            resp = client.post(f"{self.base_url}/on_enable")
        return resp.status_code == 200
    except httpx.ConnectError:
        return False


RemoteCapabilityProxy.on_disable = on_disable
RemoteCapabilityProxy.on_enable = on_enable

### CR-3: Typed MonitorToolProtocol accessors

`get_system_status` / `list_processes` POST to typed worker endpoints (`/get_system_status`, `/list_processes`) instead of the magic-string-dispatched `/execute("get_system_status")`. Both expose sync + async variants matching the existing CR-2 hook style.

**Status code taxonomy** (intentional, per CR-3 review):

| Code | Meaning | Proxy behavior |
|------|---------|----------------|
| 200 | Typed call succeeded | Return JSON body |
| 404 | Capability is not a monitor | Propagate `HTTPStatusError` (configuration error — substrate logs at WARN) |
| 500 | Real capability failure | Propagate `HTTPStatusError` (don't silently mask) |
| ConnectError | Worker may have died | Return `None` (substrate degrades to empty stats) |

The 404 path prevents masking the "wrong capability wired as `system_monitor`" misconfiguration (e.g. a non-monitor capability registered as the system monitor).

In [ ]:
#| export
def get_system_status(self) -> Optional[Dict[str, Any]]:  # SystemStats dict, or None on transport / config failure
    """CR-3: typed MonitorToolProtocol accessor. POSTs to worker's `/get_system_status`.
    
    Status code semantics (worker side raises HTTPException with these codes):
    - 200: SystemStats dict returned
    - 404: capability is not a monitor — logged at ERROR (configuration error;
          no amount of retry fixes it) and returns None. Loudly distinguished
          from the substrate's WARN-level transient-failure degradation.
    - 500: real capability failure; propagates as HTTPStatusError
    - ConnectError: worker may have died; returns None silently (substrate
          degrades to empty stats)
    """
    try:
        with httpx.Client(timeout=5) as client:
            resp = client.post(f"{self.base_url}/get_system_status")
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code == 404:
            # CR-3 follow-up: loud misconfiguration log. The substrate's
            # _get_global_stats catches the None return and degrades to {},
            # so a single ERROR here (not WARN at the substrate layer) gives
            # operators an actionable signal without double-logging.
            _logger.error(
                "Capability %r registered as system_monitor lacks the MonitorToolProtocol "
                "interface (HTTP 404 from /get_system_status): %s",
                self.name, resp.text,
            )
            return None
        resp.raise_for_status()
        return None
    except httpx.ConnectError:
        return None

RemoteCapabilityProxy.get_system_status = get_system_status

In [ ]:
#| export
async def get_system_status_async(self) -> Optional[Dict[str, Any]]:  # SystemStats dict, or None on transport / config failure
    """Async variant of `get_system_status`. Same 200/404/500/ConnectError semantics."""
    try:
        async with httpx.AsyncClient(timeout=5) as client:
            resp = await client.post(f"{self.base_url}/get_system_status")
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code == 404:
            # CR-3 follow-up: loud misconfiguration log (see sync variant)
            _logger.error(
                "Capability %r registered as system_monitor lacks the MonitorToolProtocol "
                "interface (HTTP 404 from /get_system_status): %s",
                self.name, resp.text,
            )
            return None
        resp.raise_for_status()
        return None
    except httpx.ConnectError:
        return None

RemoteCapabilityProxy.get_system_status_async = get_system_status_async

In [ ]:
#| export
def list_processes(self) -> Optional[List[Dict[str, Any]]]:  # ProcessStats dict list, or None on transport / config failure
    """CR-3: typed MonitorToolProtocol accessor. POSTs to worker's `/list_processes`.
    
    Same 200/404/500/ConnectError semantics as `get_system_status`. Note that
    `MonitorToolProtocol.list_processes()` defaults to returning `[]`, so monitors without
    per-process visibility yield a 200 with an empty list.
    """
    try:
        with httpx.Client(timeout=5) as client:
            resp = client.post(f"{self.base_url}/list_processes")
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code == 404:
            # CR-3 follow-up: loud misconfiguration log
            _logger.error(
                "Capability %r registered as system_monitor lacks the MonitorToolProtocol "
                "interface (HTTP 404 from /list_processes): %s",
                self.name, resp.text,
            )
            return None
        resp.raise_for_status()
        return None
    except httpx.ConnectError:
        return None

RemoteCapabilityProxy.list_processes = list_processes

In [ ]:
#| export
async def list_processes_async(self) -> Optional[List[Dict[str, Any]]]:  # ProcessStats dict list, or None on transport / config failure
    """Async variant of `list_processes`. Same semantics."""
    try:
        async with httpx.AsyncClient(timeout=5) as client:
            resp = await client.post(f"{self.base_url}/list_processes")
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code == 404:
            # CR-3 follow-up: loud misconfiguration log
            _logger.error(
                "Capability %r registered as system_monitor lacks the MonitorToolProtocol "
                "interface (HTTP 404 from /list_processes): %s",
                self.name, resp.text,
            )
            return None
        resp.raise_for_status()
        return None
    except httpx.ConnectError:
        return None

RemoteCapabilityProxy.list_processes_async = list_processes_async

### CR-4: Lifecycle hook proxies (prefetch + reconfigure)

`prefetch()` / `reconfigure()` POST to the worker's `/prefetch` and `/reconfigure` endpoints. Both expose sync + async variants matching the existing CR-2/CR-3 hook style.

| Endpoint | Proxy method | Behavior |
|----------|--------------|----------|
| `/prefetch` | `prefetch()` / `prefetch_async()` | Returns `True` on 200, `False` on transport failure. Errors raised by the capability's `prefetch()` (worker 500) propagate as `RuntimeError` so callers can distinguish "capability can't acquire resources" from "worker unreachable." |
| `/reconfigure` | `reconfigure(old, new)` / `reconfigure_async(old, new)` | Returns `True` on 200, `False` on transport failure. Body shape: `{"old_config": ..., "new_config": ...}`. Capability's default `reconfigure` body delegates to `reconfigure_with_triggers`, which walks `RELOAD_TRIGGER` metadata to fire `_release_<trigger>` methods. |

The substrate's `CapabilityManager.update_capability_config` path (CR-2 onward) calls `reconfigure(old, new)` when the worker exposes the typed endpoint. Older workers fall back to `initialize(new_config)` — both paths converge on the same worker-side state after the call.

In [ ]:
#| export
def prefetch(
    self,
    stall_threshold_seconds: Optional[float] = None,  # Override SubstrateConfig.prefetch_stall_threshold_seconds; None = use config
    poll_interval_seconds: float = 1.0,               # How often to poll /progress for stall detection
) -> bool:  # True if worker accepted the prefetch hook
    """CR-4 / Session A 2026-05-27: forward the substrate's prefetch signal with
    progress-based stall detection.

    Replaces wall-clock-timeout-based startup waiting (operators racing arbitrary
    timeouts vs. network speeds for model downloads). Approach:

      1. POST /prefetch fires in a background thread with httpx timeout=None.
      2. Main thread polls /progress every poll_interval_seconds.
      3. Each (progress, message) change resets the stall counter.
      4. If no change in stall_threshold_seconds AND POST still pending →
         SIGTERM the worker subprocess + raise CapabilityTimeoutError.

    Capabilities opt in to fine-grained stall defeat by calling
    self.report_progress(...) periodically during long lifecycle operations
    (model download, server startup, etc.). Capabilities that don't report progress
    are fine as long as the threshold accommodates their slowest plausible
    silent stretch.

    Errors raised by the capability (worker 500) propagate as RuntimeError; worker
    unreachable propagates as `False`; stall fires CapabilityTimeoutError.
    """
    threshold = stall_threshold_seconds if stall_threshold_seconds is not None else _resolve_prefetch_stall_threshold()
    return _run_prefetch_with_stall_detection(self, threshold, poll_interval_seconds)

RemoteCapabilityProxy.prefetch = prefetch

In [ ]:
#| export
async def prefetch_async(
    self,
    stall_threshold_seconds: Optional[float] = None,
    poll_interval_seconds: float = 1.0,
) -> bool:  # True if worker accepted the prefetch hook
    """Async variant of `prefetch`. Same stall-detection semantics."""
    threshold = stall_threshold_seconds if stall_threshold_seconds is not None else _resolve_prefetch_stall_threshold()
    return await _run_prefetch_with_stall_detection_async(self, threshold, poll_interval_seconds)

RemoteCapabilityProxy.prefetch_async = prefetch_async

In [ ]:
#| export
def _resolve_prefetch_stall_threshold() -> float:
    """Resolve the stall threshold from SubstrateConfig with a defensive fallback."""
    try:
        from cjm_substrate.core.config import get_config
        cfg = get_config()
        return float(getattr(cfg.substrate, 'prefetch_stall_threshold_seconds', 60.0))
    except Exception:
        return 60.0

In [ ]:
#| export
def _run_prefetch_with_stall_detection(
    proxy: 'RemoteCapabilityProxy',
    stall_threshold_seconds: float,
    poll_interval_seconds: float,
) -> bool:
    """Sync stall-detecting prefetch implementation.

    Runs POST /prefetch in a daemon thread; main thread polls /progress for
    a (progress, message) advance every poll_interval_seconds. If no advance
    in stall_threshold_seconds AND the POST is still in-flight, SIGTERMs the
    worker subprocess (so its capability.cleanup() can run via the worker's
    shutdown handler — closes the orphan-subprocess-on-stall bug) and raises
    CapabilityTimeoutError client-side.
    """
    import threading
    import time

    from cjm_substrate.core.errors import CapabilityTimeoutError

    state: Dict[str, Any] = {'status': 'pending', 'error': None}

    def _post_prefetch() -> None:
        try:
            with httpx.Client(timeout=None) as client:
                resp = client.post(f"{proxy.base_url}/prefetch")
            if resp.status_code == 500:
                state['error'] = RuntimeError(f"Capability prefetch failed: {resp.text}")
            elif resp.status_code == 200:
                state['status'] = 'done'
            else:
                state['status'] = 'unexpected_status'
        except httpx.ConnectError:
            state['status'] = 'connect_error'
        except Exception as e:
            state['error'] = e

    t = threading.Thread(target=_post_prefetch, daemon=True, name=f"prefetch-{proxy.name}")
    t.start()

    last_progress: Tuple[float, str] = (-1.0, '__sentinel__')
    last_change = time.time()
    while t.is_alive():
        t.join(timeout=poll_interval_seconds)
        if not t.is_alive():
            break
        # Poll progress to defeat stall counter.
        try:
            with httpx.Client(timeout=2.0) as poll_client:
                pr = poll_client.get(f"{proxy.base_url}/progress")
            if pr.status_code == 200:
                pd = pr.json()
                current = (float(pd.get('progress', 0.0) or 0.0), str(pd.get('message', '')))
                if current != last_progress:
                    last_progress = current
                    last_change = time.time()
        except Exception:
            # Polling failure is non-fatal; the POST thread carries the real signal.
            pass

        if time.time() - last_change > stall_threshold_seconds:
            # Stall: terminate the worker + its full process subtree via
            # platform.terminate_process (process-group SIGTERM + psutil safety
            # sweep). The worker's @on_event("shutdown") gets a best-effort
            # chance to fire capability.cleanup() during graceful shutdown; the
            # subtree-kill catches any subprocess (vLLM api_server, EngineCore,
            # etc.) that the shutdown path missed. Re-raise client-side.
            try:
                from cjm_substrate.core.platform import terminate_process as _tp
                _tp(proxy.process, timeout=3.0)
            except Exception:
                pass
            elapsed = time.time() - last_change
            raise CapabilityTimeoutError(proxy.name, elapsed)

    if state['error'] is not None:
        raise state['error']
    if state['status'] == 'connect_error':
        return False  # Worker died
    return state['status'] == 'done'

In [ ]:
#| export
async def _run_prefetch_with_stall_detection_async(
    proxy: 'RemoteCapabilityProxy',
    stall_threshold_seconds: float,
    poll_interval_seconds: float,
) -> bool:
    """Async stall-detecting prefetch implementation. Mirrors the sync variant
    using asyncio.gather instead of a daemon thread."""
    import asyncio
    import time

    from cjm_substrate.core.errors import CapabilityTimeoutError

    async def _post_prefetch() -> Dict[str, Any]:
        try:
            async with httpx.AsyncClient(timeout=None) as client:
                resp = await client.post(f"{proxy.base_url}/prefetch")
            if resp.status_code == 500:
                return {'status': 'capability_error', 'detail': resp.text}
            if resp.status_code == 200:
                return {'status': 'done'}
            return {'status': 'unexpected_status', 'detail': resp.status_code}
        except httpx.ConnectError:
            return {'status': 'connect_error'}

    async def _poll_progress() -> None:
        nonlocal last_progress, last_change
        try:
            async with httpx.AsyncClient(timeout=2.0) as client:
                pr = await client.get(f"{proxy.base_url}/progress")
            if pr.status_code == 200:
                pd = pr.json()
                current = (float(pd.get('progress', 0.0) or 0.0), str(pd.get('message', '')))
                if current != last_progress:
                    last_progress = current
                    last_change = time.time()
        except Exception:
            pass

    last_progress: Tuple[float, str] = (-1.0, '__sentinel__')
    last_change = time.time()
    post_task = asyncio.create_task(_post_prefetch())
    while not post_task.done():
        try:
            await asyncio.wait_for(asyncio.shield(post_task), timeout=poll_interval_seconds)
        except asyncio.TimeoutError:
            pass
        if post_task.done():
            break
        await _poll_progress()
        if time.time() - last_change > stall_threshold_seconds:
            post_task.cancel()
            try:
                from cjm_substrate.core.platform import terminate_process as _tp
                _tp(proxy.process, timeout=3.0)
            except Exception:
                pass
            elapsed = time.time() - last_change
            raise CapabilityTimeoutError(proxy.name, elapsed)

    result = await post_task
    if result['status'] == 'capability_error':
        raise RuntimeError(f"Capability prefetch failed: {result['detail']}")
    if result['status'] == 'connect_error':
        return False
    return result['status'] == 'done'

In [ ]:
#| export
def reconfigure(
    self,
    old_config: Optional[Dict[str, Any]],  # Previous config snapshot
    new_config: Optional[Dict[str, Any]],  # Config being applied
) -> bool:  # True if worker accepted the reconfigure call
    """CR-4: forward a reconfigure(old, new) call to the worker process.
    
    The capability's default reconfigure() body delegates to
    reconfigure_with_triggers, which walks RELOAD_TRIGGER metadata on the
    capability's config_class to fire `_release_<trigger>` methods for fields
    whose values changed. Capabilities not opting into the declarative pattern
    land in a silent no-op; the substrate's CapabilityManager.update_capability_config
    then falls back to initialize(new_config) for the actual state change.
    """
    payload = {"old_config": old_config, "new_config": new_config}
    try:
        with httpx.Client(timeout=None) as client:
            resp = client.post(f"{self.base_url}/reconfigure", json=payload)
        if resp.status_code == 500:
            raise RuntimeError(f"Capability reconfigure failed: {resp.text}")
        return resp.status_code == 200
    except httpx.ConnectError:
        return False

RemoteCapabilityProxy.reconfigure = reconfigure

In [ ]:
#| export
async def reconfigure_async(
    self,
    old_config: Optional[Dict[str, Any]],  # Previous config snapshot
    new_config: Optional[Dict[str, Any]],  # Config being applied
) -> bool:  # True if worker accepted the reconfigure call
    """Async variant of `reconfigure`. Same semantics."""
    payload = {"old_config": old_config, "new_config": new_config}
    try:
        async with httpx.AsyncClient(timeout=None) as client:
            resp = await client.post(f"{self.base_url}/reconfigure", json=payload)
        if resp.status_code == 500:
            raise RuntimeError(f"Capability reconfigure failed: {resp.text}")
        return resp.status_code == 200
    except httpx.ConnectError:
        return False

RemoteCapabilityProxy.reconfigure_async = reconfigure_async

### Context Manager Support

The proxy can be used as a context manager for automatic cleanup.

In [ ]:
#| export
def __enter__(self):
    """Enter context manager."""
    return self

def __exit__(self, exc_type, exc_val, exc_tb):
    """Exit context manager and cleanup."""
    self.cleanup()
    return False

async def __aenter__(self):
    """Enter async context manager."""
    return self

async def __aexit__(self, exc_type, exc_val, exc_tb):
    """Exit async context manager and cleanup."""
    self.cleanup()
    return False

RemoteCapabilityProxy.__enter__ = __enter__
RemoteCapabilityProxy.__exit__ = __exit__
RemoteCapabilityProxy.__aenter__ = __aenter__
RemoteCapabilityProxy.__aexit__ = __aexit__

## Usage Examples

### Basic Usage (Sync)

```python
# Load manifest from JSON file
with open("~/.cjm/manifests/whisper.json") as f:
    manifest = json.load(f)

# Create proxy (starts worker subprocess)
capability = RemoteCapabilityProxy(manifest)

# Use like a local capability
capability.initialize({"model": "large-v3"})
result = capability.execute(audio="/path/to/audio.wav")

# Clean up
capability.cleanup()
```

### With Context Manager

```python
with RemoteCapabilityProxy(manifest) as capability:
    capability.initialize({"model": "large-v3"})
    result = capability.execute(audio="/path/to/audio.wav")
# Worker automatically terminated
```

### Async with Streaming (FastHTML)

```python
async def transcribe_stream(audio_path: str):
    async with RemoteCapabilityProxy(manifest) as capability:
        await capability.initialize({"model": "large-v3"})
        async for chunk in capability.execute_stream(audio=audio_path):
            yield chunk  # Stream to client
```

## Manifest Format

The proxy expects a manifest dictionary with at minimum:

```json
{
    "name": "whisper-local",
    "version": "1.0.0",
    "python_path": "/home/user/anaconda3/envs/cjm-whisper/bin/python",
    "module": "cjm_transcription_capability_whisper.capability",
    "class": "WhisperLocalCapability",
    "env_vars": {
        "CUDA_VISIBLE_DEVICES": "0"
    }
}
```

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()